# **UBER PROJECT - Clustering models** #

After analyzing the uber datasets and cleaning the data, we can now create clusters by using two models: Kmeans and DBSCAN.

For the clustering models we are going to use the dataset from 2015 that gathers rides per taxi zones and not directly each pick up zone. This will facilitate the clustering process, moreover we have already seen in the EDA phase that there were no huge differences between the 2014 and 2015 patterns.

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

## **I. Kmeans** ##

### **1. Identification of K with the Elbow and Silhouette methods** ###

In order to use the Kmeans model, we need to identify K (the number of cluster) in advance, to select the best K, we can use the Elbow and Silhouette methods.

In [12]:
uber_2015= pd.read_csv(
    "../DATA/DATA_created/uber_2015_final_for_modelling.csv",
      usecols=['Latitude', 'Longitude', 'Zone', 'Borough', 'Hour', 'Weekday']
)

In [ ]:
print(f"Total rides: {len(uber_2015):,}")

Total rides: 493,263


In [ ]:
# Step 1 : Extract geographic coordinates
coordinates = uber_2015[['Latitude', 'Longitude']].values


In [6]:
# Step 2 : Standardize the data (because latitude and longitude have different scales) 
scaler = StandardScaler()

# Fit and transform the coordinates
coordinates_scaled = scaler.fit_transform(coordinates)

In [6]:
# Step 3 : ELBOW METHOD:

# Define range of K values to test
K_range = range(2, 11)  # Test K from 2 to 10 clusters

# Store inertia values
wcss =  []
inertias = []

for k in K_range:
    # Train KMeans with current K
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',  # Better initialization
        n_init=10,         # Try 10 times with different initializations
        random_state=42,   # For reproducibility
    )
    
    # Fit the model
    kmeans.fit(coordinates_scaled)
    
    # Store inertia
    inertias.append(kmeans.inertia_)
    
# Create DataFrame
frame = pd.DataFrame({
    "k": list(K_range),
    "inertia": inertias
})

print(frame)
fig = px.line(
    frame,
    x="k",
    y="inertia"
)

fig.update_layout(
    yaxis_title="Inertia",
    xaxis_title="Number of clusters",
    title="Elbow Method"
)

fig.show()

    k        inertia
0   2  601860.243672
1   3  389794.919318
2   4  262606.744263
3   5  189576.309359
4   6  151702.310854
5   7  129941.715422
6   8  113329.705302
7   9   97975.313689
8  10   82018.569866


In [7]:
# As the silhouette score is established on a sampled dataset, we will try several options and calculate the mean.
def compute_silhouette(df, K_range, n_runs=5, sample_size=15000):
    
    X = df[['Latitude', 'Longitude']].values
    
    all_scores = []

    for _ in range(n_runs):
        idx = np.random.choice(len(X), sample_size, replace=False)
        sample = X[idx]

        scores = []

        for k in K_range:
            kmeans = KMeans(n_clusters=k, random_state=42)
            labels = kmeans.fit_predict(sample)
            score = silhouette_score(sample, labels)
            scores.append(score)

        all_scores.append(scores)

    return np.mean(all_scores, axis=0)

In [ ]:
compute_silhouette(uber_2015,range(2, 11),n_runs=5,sample_size=15000)

array([0.67314608, 0.48505979, 0.42186652, 0.43057656, 0.45400824,
       0.46084753, 0.48074118, 0.47226692, 0.45792462])

The elbow method suggests an optimal number of clusters around K=4 or K=5, where the decrease in inertia starts to slow down significantly. Meanwhile, the silhouette score stabilizes around K=4, indicating a good balance between cluster separation and cohesion. Therefore, K=4 is selected as the optimal number of clusters, providing meaningful and interpretable geographic segmentation.

### **2. Kmeans model** ###

Now that we have identified the best K for our study, we can implement a Kmeans model.

In [9]:
K=4
kmeans = KMeans(
    n_clusters=K,           
    init='k-means++',       # Smart initialization
    n_init=10,              # Run 10 times, we keep the best
    max_iter=300, 
    random_state=42        
)

In [10]:
cluster_labels = kmeans.fit_predict(coordinates_scaled)
print(f"  Inertia (sum of squared distances): {kmeans.inertia_:.2f}")
print(f"  Number of iterations: {kmeans.n_iter_}")

  Inertia (sum of squared distances): 262606.74
  Number of iterations: 6


In [ ]:
# Distribution per cluster
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
 
print("\nCluster distribution:")
for cluster_id in range(K):
    count = cluster_counts[cluster_id]
    pct = 100 * count / len(cluster_labels)
    print(f"  Cluster {cluster_id}: {count:,} rides ({pct:.1f}%)")
 
# Get centers
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
 
print("\nCluster centers:")
for cluster_id in range(K):
    lat, lon = centers_original[cluster_id]
    print(f"  Cluster {cluster_id}: ({lat:.4f}, {lon:.4f})")

#  Add cluster labels
 
uber_2015['Cluster'] = cluster_labels



Cluster distribution:
  Cluster 0: 48,780 rides (9.9%)
  Cluster 1: 361,836 rides (73.4%)
  Cluster 2: 23,652 rides (4.8%)
  Cluster 3: 58,995 rides (12.0%)

Cluster centers:
  Cluster 0: (40.7893, -73.9030)
  Cluster 1: (40.7405, -73.9865)
  Cluster 2: (40.6694, -73.7928)
  Cluster 3: (40.6719, -73.9682)


In [ ]:
# Analyze by borough
print(f"\nBorough distribution by cluster:")
borough_cluster = pd.crosstab(
    uber_2015['Cluster'], 
    uber_2015['Borough'],
    margins=True
)
 
print(borough_cluster)
 
print(f"\nTop zones per cluster:")
for cluster_id in range(K):
    cluster_data = uber_2015[uber_2015['Cluster'] == cluster_id]
    top_zones = cluster_data['Zone'].value_counts().head(5)
    
    print(f"\n  Cluster {cluster_id}:")
    for zone, count in top_zones.items():
        pct = 100 * count / len(cluster_data)
        print(f"    {zone}: {count:,} ({pct:.1f}%)")
 


Borough distribution by cluster:
Borough  Bronx  Brooklyn  Manhattan  Queens  Staten Island     All
Cluster                                                           
0         5973         0      13742   29065              0   48780
1            0     20904     332526    8406              0  361836
2            0       105          0   23547              0   23652
3            0     57195          0    1685            115   58995
All       5973     78204     346268   62703            115  493263

Top zones per cluster:

  Cluster 0:
    LaGuardia Airport: 13,226 (27.1%)
    Astoria: 4,525 (9.3%)
    Central Harlem: 3,149 (6.5%)
    Manhattan Valley: 2,907 (6.0%)
    Morningside Heights: 2,859 (5.9%)

  Cluster 1:
    Midtown Center: 22,469 (6.2%)
    Union Sq: 20,617 (5.7%)
    TriBeCa/Civic Center: 20,546 (5.7%)
    East Village: 20,024 (5.5%)
    West Village: 15,831 (4.4%)

  Cluster 2:
    JFK Airport: 13,958 (59.0%)
    Forest Hills: 2,029 (8.6%)
    Jamaica: 787 (3.3%)
    Hill

In [ ]:
# Analyze by hour
 
print(f"\nPickup hour preference by cluster:")
for cluster_id in range(K):
    cluster_data = uber_2015[uber_2015['Cluster'] == cluster_id]
    peak_hour = cluster_data['Hour'].mode()[0]
    avg_hour = cluster_data['Hour'].mean()
    
    print(f"Cluster {cluster_id}:")
    print(f"    - Peak hour: {peak_hour}:00")
    print(f"    - Average hour: {avg_hour:.1f}")


Pickup hour preference by cluster:
Cluster 0:
    - Peak hour: 21:00
    - Average hour: 13.9
Cluster 1:
    - Peak hour: 19:00
    - Average hour: 14.3
Cluster 2:
    - Peak hour: 22:00
    - Average hour: 13.6
Cluster 3:
    - Peak hour: 22:00
    - Average hour: 13.5


In [ ]:
# Save main results
uber_2015.to_csv("../DATA/DATA_Kmeans/uber_2015_k4_final.csv", index=False)

 
# Save centers
centers_df = pd.DataFrame({
    'Cluster': range(K),
    'Center_Latitude': centers_original[:, 0],
    'Center_Longitude': centers_original[:, 1],
    'Rides_Count': [cluster_counts[i] for i in range(K)],
    'Rides_Percent': [f"{100*cluster_counts[i]/len(cluster_labels):.1f}%" for i in range(K)]
})
 
centers_df.to_csv("../DATA/DATA_Kmeans/kmeans_k4_centers.csv", index=False)


We can now visualize the results of this clustering.

In [23]:
colors = {0: '#FF6B6B', 1: '#4ECDC4', 2: "#13B824", 3: "#F9A704"}
cluster_names = {
    0: 'LaGuardia/Queens/Manhattan',
    1: 'Manhattan Center',
    2: 'JFK/Queens',
    3: 'Brooklyn'
}
 

In [ ]:
fig = go.Figure()
 
# Add traces for each cluster
for cluster_id in sorted(uber_2015['Cluster'].unique()):
    cluster_data = uber_2015[uber_2015['Cluster'] == cluster_id]
    center_row = centers_df[centers_df['Cluster'] == cluster_id].iloc[0]
    
    # Sample points for better performance (10k per cluster)
    sample_size = min(10000, len(cluster_data))
    cluster_sample = cluster_data.sample(n=sample_size, random_state=42)
    
    # Add cluster points
    fig.add_trace(go.Scattermapbox(
        lon=cluster_sample['Longitude'],
        lat=cluster_sample['Latitude'],
        mode='markers',
        marker=dict(
            size=10,
            opacity=0.65,
            color=colors[cluster_id],
        ),
        text=cluster_sample['Zone'],
        customdata=cluster_sample[['Borough', 'Hour']],
        hovertemplate='<b>%{text}</b><br>' +
                      'Borough: %{customdata[0]}<br>' +
                      'Hour: %{customdata[1]}:00<br>' +
                      '<extra></extra>',
        name=f'Cluster {cluster_id}: {cluster_names[cluster_id]}',
        visible=True,  # All visible by default
        legendgroup=f'cluster{cluster_id}'
    ))
    
    # Add cluster center
    fig.add_trace(go.Scattermapbox(
        lon=[center_row['Center_Longitude']],
        lat=[center_row['Center_Latitude']],
        mode='markers+text',
        marker=dict(
            size=25,
            color=colors[cluster_id],
            symbol='circle',
        ),
        text=[f'C{cluster_id}'],
        textposition='top center',
        textfont=dict(size=14, color='white', family='Arial Black'),
        hovertemplate=f'<b>Cluster {cluster_id} Center</b><br>' +
                      f'{cluster_names[cluster_id]}<br>' +
                      f'Total Rides: {int(center_row["Rides_Count"]):,}<br>' +
                      f'Percentage: {center_row["Rides_Percent"]}<br>' +
                      f'<extra></extra>',
        name=f'Cluster {cluster_id} CENTER',
        visible=True,
        legendgroup=f'cluster{cluster_id}',
        showlegend=False
    ))
 
fig.update_layout(
    mapbox=dict(
        style='carto-positron',  # Clean, simple Google Maps-like style
        center=dict(lat=40.7128, lon=-74.0060),
        zoom=10.5,
        bearing=0,
        pitch=0
    ),
    
    # Size and title
    height=800,
    width=1400,
    title=dict(
        text='<b>Uber NYC Hotspots - KMeans K=4</b><br><sub>Click legend to toggle clusters</sub>',
        x=0.5,
        xanchor='center',
        font=dict(size=20)
    ),
    
    # Legend styling
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='gray',
        borderwidth=2,
        font=dict(size=12)
    ),
    
    # Hover mode
    hovermode='closest',
    
    # Margins
    margin=dict(r=0, t=80, l=0, b=0),
    
    # Responsive
    autosize=False
)

fig.write_html("../DATA/DATA_Kmeans/clusters_map.html")

### **II. DBSCAN** ###

For this model we do not need to determine K in advance but we need to determine : 
- `eps` : the maximum distance between points,
- `min_samples` : the minimum number of points in neighborhood.

### **1. Find optimal `eps`and `minsamples`** ###

First we need to aggregate rides per zone (because many rides have identical coordinates).

In [18]:
zone_aggregated = uber_2015.groupby(['Latitude', 'Longitude']).agg({
    'Zone': 'first',
    'Borough': 'first',
    'Hour': 'first'
}).reset_index()
 
zone_aggregated['Pickup_Count'] = uber_2015.groupby(['Latitude', 'Longitude']).size().values
 
print(f"\nOriginal rides: {len(uber_2015):,}")
print(f"Unique zone locations: {len(zone_aggregated)}")


Original rides: 493,263
Unique zone locations: 180


In [19]:
# Normalize
coordinates = zone_aggregated[['Latitude', 'Longitude']].values
scaler = StandardScaler()
coordinates_scaled = scaler.fit_transform(coordinates)

In [24]:
# Create range of eps values to test
eps_values = np.arange(0.05, 1.0, 0.05) 
 
print(f"{'EPS':<8} {'Clusters':<12} {'Noise':<10} {'Largest':<10} {'Good?'}")

 
results = []
 
for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(coordinates_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    largest = np.bincount(labels[labels >= 0]).max() if np.any(labels >= 0) else 0
    
    is_good = "yes" if 3 <= n_clusters <= 7 else ""
    
    print(f"{eps:<8.2f} {n_clusters:<12} {n_noise:<10} {largest:<10} {is_good}")
    
    results.append({
        'eps': eps,
        'clusters': n_clusters,
        'noise': n_noise,
        'largest': largest,
        'labels': labels
    })


EPS      Clusters     Noise      Largest    Good?
0.05     0            180        0          
0.10     2            168        7          
0.15     2            143        29         
0.20     5            117        34         yes
0.25     5            80         66         yes
0.30     7            42         70         yes
0.35     2            25         145        
0.40     1            19         161        
0.45     1            13         167        
0.50     1            13         167        
0.55     2            7          168        
0.60     2            6          170        
0.65     1            6          174        
0.70     1            4          176        
0.75     1            3          177        
0.80     1            3          177        
0.85     1            3          177        
0.90     1            0          180        
0.95     1            0          180        


We can select `eps`=0,30 according to this analyze. We then have 7 clusters (enough but not too much) with less noise.

In [28]:
# DBSCAN model
eps = 0.30
min_samples = 5

 
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
cluster_labels = dbscan.fit_predict(coordinates_scaled)
 

In [29]:
# Count clusters
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
 
print(f"\nResults:")
print(f"  Number of clusters found: {n_clusters}")
print(f"  Noise points (outliers): {n_noise:,} ({n_noise/len(cluster_labels)*100:.1f}%)")
 
# Cluster distribution
print(f"\nCluster sizes:")
unique_labels, counts = np.unique(cluster_labels, return_counts=True)
 
for label, count in sorted(zip(unique_labels, counts)):
    if label == -1:
        print(f"  Noise (outliers): {count:,} points")
    else:
        pct = 100 * count / len(cluster_labels)
        print(f"  Cluster {label}: {count:,} points ({pct:.1f}%)")


Results:
  Number of clusters found: 7
  Noise points (outliers): 42 (23.3%)

Cluster sizes:
  Noise (outliers): 42 points
  Cluster 0: 8 points (4.4%)
  Cluster 1: 70 points (38.9%)
  Cluster 2: 7 points (3.9%)
  Cluster 3: 12 points (6.7%)
  Cluster 4: 7 points (3.9%)
  Cluster 5: 5 points (2.8%)
  Cluster 6: 29 points (16.1%)


In [30]:
zone_aggregated['Cluster_DBSCAN'] = cluster_labels
 
# Map zones back to original rides
zone_to_cluster = dict(zip(
    zone_aggregated.index,
    cluster_labels
))
 
# Create mapping from (lat, lon) to cluster
coord_to_cluster = {}
for idx, row in zone_aggregated.iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    coord_to_cluster[(lat, lon)] = cluster_labels[idx]
 
# Add to original data
uber_2015['Cluster_DBSCAN'] = uber_2015.apply(
    lambda row: coord_to_cluster.get((row['Latitude'], row['Longitude']), -1),
    axis=1
)
 

In [ ]:

colors_map = {
    -1: 'gray',
    0: 'red',
    1: 'blue',
    2: 'green',
    3: 'orange',
    4: 'purple',
    5: 'brown',
    6: 'black',
    7: 'yellow'
}
 
fig = go.Figure()
 
# Add cluster points
for cluster_id in sorted(set(cluster_labels)):
    mask = cluster_labels == cluster_id
    cluster_zones = zone_aggregated[mask]
    
    if cluster_id == -1:
        name = f'Noise/Outliers ({len(cluster_zones)} zones)'
        opacity = 0.3
        size = 6
    else:
        total_rides = cluster_zones['Pickup_Count'].sum()
        name = f'Cluster {cluster_id} ({len(cluster_zones)} zones, {total_rides:,} rides)'
        opacity = 0.7
        size = 10
    
    fig.add_trace(go.Scattermapbox(
        lon=cluster_zones['Longitude'],
        lat=cluster_zones['Latitude'],
        mode='markers',
        marker=dict(
            size=size,
            opacity=opacity,
            color=colors_map.get(cluster_id, 'gray'),
        ),
        text=cluster_zones['Zone'],
        customdata=cluster_zones['Pickup_Count'],
        hovertemplate='<b>%{text}</b><br>' +
                      'Pickups: %{customdata:,}<extra></extra>',
        name=name,
        showlegend=True
    ))
 
fig.update_layout(
    mapbox=dict(
        style='carto-positron',
        center=dict(lat=40.7128, lon=-74.0060),
        zoom=10.5
    ),
    height=800,
    width=1400,
    title=f'DBSCAN Clustering on Zone Aggregated Data - {n_clusters} Clusters',
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='gray',
        borderwidth=2,
        font=dict(size=11)
    ),
    hovermode='closest',
    margin=dict(r=0, t=80, l=0, b=0)
)
 
fig.write_html("dbscan_aggregated_map.html")
fig.show()

Saved: dbscan_aggregated_map.html
